# Arduino 4DOF Robot Arm: Engineering Notebook

This notebook is the technical knowledge base for the project. The embedded implementation is **Arduino C++**; Python cells are supplemental calculations only.

Computational topics follow a consistent progression: **theory and mathematical model → Arduino C++ implementation → numerical or visual result → interpretation**. Documentary sections such as hardware, limitations, conclusions, and reproduction instructions retain a conventional reference structure.

**Evidence labels used throughout**

- **Source-verified** — read directly from the repository.
- **Vendor-reported** — stated by the Keyestudio tutorial.
- **Physical evidence** — visible in the supplied demonstration media.
- **Calculated** — derived from source constants or algorithms.
- **Illustrative** — synthetic values used only to explain behavior.
- **Uncertain** — cannot be established from the available evidence.


## 1. Problem Definition and Scope

The system controls a desktop robot arm through four commanded degrees of freedom: base rotation, two arm-linkage servos, and a gripper servo. Two analog joysticks provide manual commands, while their push buttons capture and replay servo setpoints.

### Requirements established by the source

1. Initialize four servos to a mechanically usable boot posture.
2. Read four analog axes and two digital buttons.
3. Increment or decrement joint commands inside calibrated limits.
4. Store one pose or a bounded sequence of poses.
5. Replay saved poses in one-degree increments.
6. Preserve Arduino C++ as the implementation language.

The repository does **not** implement encoder feedback, inverse kinematics, payload sensing, collision detection, or closed-loop Cartesian control.


## 2. System Architecture

![System architecture](../assets/diagrams/system-architecture.svg)

*Figure 1. Project-specific signal, memory, actuator, and power architecture.*

The control chain is: operator input → analog/digital acquisition → threshold and limit logic → commanded servo angle → Servo library pulse generation → driver shield → mechanical actuator. The optional memory path stores the commanded angle vector and feeds it back to the same output layer during replay.


## 3. Hardware and Electrical Interfaces

| Element | Verified or reported role |
|---|---|
| Keyestudio V4.0 board | Arduino Uno-compatible ATmega328P controller with CP2102 USB-to-UART |
| TB6612FNG motor/servo shield | Servo connectors, regulated 5 V distribution, and 7–15 V VIN input |
| Four 180° servos | Base, two arm-linkage motions, and gripper |
| Two joystick modules | Four analog axes and two digital push buttons |
| Mechanical kit | Acrylic structure, linkage hardware, fasteners, and battery holder |

![Keyestudio core components](../assets/reference/keyestudio-core-components.png)

*Figure 2. Vendor reference - controller board, TB6612FNG shield, and acrylic arm components (Keyestudio KS0198 tutorial, Kit List).*

The vendor's numbered controller-board illustration identifies: 1 ICSP, 2 power LED, 3 digital I/O, 4 GND, 5 AREF, 6 SDA, 7 SCL, 8 reset button, 9 D13 LED, 10 USB, 11 CP2102, 12 TX LED, 13 RX LED, 14 crystal oscillator, 15 voltage regulator, 16 DC jack, 17 microcontroller, 18 IOREF, 19 reset header, 20 3.3 V, 21 5 V, 22 VIN, and 23 analog inputs.

> Safety: servo current must be supplied through the driver shield with common ground. The shield VIN rating must not be confused with a direct 5 V servo connection.


### 3.1 Source-verified pin map

![Pin map](../assets/diagrams/pin-map.svg)

*Figure 3. Source-verified electrical pin and power map.*

| Function | Pin |
|---|---:|
| Servo 1 — base | A1 |
| Servo 2 — right-side arm | A0 |
| Servo 3 — left-side arm | D6 |
| Servo 4 — gripper | D9 |
| Right joystick X / Y / button | A2 / A5 / D7 |
| Left joystick X / Y / button | A3 / A4 / D8 |


### 3.2 Physical joint map

![Physical joint and servo map](../assets/diagrams/joint-map.svg)

*Figure 4. Simplified physical map of Servo 1-4, their functional joints, signal pins, and joystick axes. Exact motion depends on horn orientation and linkage assembly.*

### 3.3 Vendor assembly and wiring references

![Servo 3 assembly reference](../assets/reference/servo-3-assembly-reference.png)

*Figure 5. Vendor assembly reference for the Servo 3 horn orientation and linkage (Keyestudio KS0198 tutorial, Assembly Guide).*

![Complete servo and joystick wiring reference](../assets/reference/servo-joystick-wiring.png)

*Figure 6. Vendor wiring overview for four servos and two joystick modules (Keyestudio KS0198 tutorial, Add Memory Function). The project pin map above remains the source-verified software reference.*


### 3.4 Source-verified joystick-to-motion map

| Input condition | Physical stick direction | Servo and joint | Angle update | Motion described by the current code |
|---|---|---|---:|---|
| `x1 < 50` | Right joystick right | Servo 1 - base | -1 degree | Rotate right |
| `x1 > 1000` | Right joystick left | Servo 1 - base | +1 degree | Rotate left |
| `y1 > 1000` | Right joystick up | Servo 2 - shoulder | -1 degree | Raise upper arm |
| `y1 < 50` | Right joystick down | Servo 2 - shoulder | +1 degree | Lower upper arm |
| `x2 < 50` | Left joystick right | Servo 4 - gripper | -2 degrees | Close gripper |
| `x2 > 1000` | Left joystick left | Servo 4 - gripper | +8 degrees | Open gripper |
| `y2 < 50` | Left joystick up | Servo 3 - elbow | +1 degree | Extend lower arm |
| `y2 > 1000` | Left joystick down | Servo 3 - elbow | -1 degree | Retract lower arm |

The right button on D7 captures a pose and the left button on D8 starts replay in the pose-memory sketches. Button behavior differs slightly in the looping sketch because a stop event must also be handled.

> **Servo 3 calibration note.** All current repository sketches use the same angle-direction mapping shown above. The vendor's Project 3.3 table assigns the opposite physical labels, extend versus retract, to those same Servo 3 angle changes. Verify the actual motion on the assembled linkage before changing code; the pin and numerical angle direction are not disputed.


## 4. Servo Pulse Theory

The low-level calibration sketches generate a high pulse whose width is linear in the requested angle:

$$
\tau(\theta)=500+11\theta \quad [\mu s], \qquad 0^\circ\leq\theta\leq180^\circ.
$$

The implementation then delays for an integer number of milliseconds:

$$
T(\theta)=\frac{\tau(\theta)}{1000}+20-\left\lfloor\frac{\tau(\theta)}{1000}\right\rfloor \quad [ms].
$$

This produces a frame rate close to 49 Hz, with small quantization from integer division.


### 4.1 Arduino C++ implementation

The calibration sketches implement the pulse equation directly in Arduino C++:

```cpp
void servopulse(int servopin, int myangle)
{
  pulsewidth = (myangle * 11) + 500;
  digitalWrite(servopin, HIGH);
  delayMicroseconds(pulsewidth);
  digitalWrite(servopin, LOW);
  delay(20 - pulsewidth / 1000);
}
```

The operational sketches use Arduino's `Servo` library instead of generating the pulse manually.


### 4.2 Numerical result

The following supplemental Python cell evaluates the exact constants used by the C++ calibration sketches. Python is used only for notebook calculation and does not control the robot.


In [1]:
# Supplemental calculation from hardware_tests/servo_*/servo_*.ino
angles = (0, 80, 180)
print(f"{'angle_deg':>10}  {'pulse_us':>8}  {'frame_ms':>8}  {'frequency_hz':>12}")
for angle in angles:
    pulse_us = angle * 11 + 500
    frame_ms = pulse_us / 1000 + 20 - pulse_us // 1000
    print(f"{angle:10d}  {pulse_us:8d}  {frame_ms:8.3f}  {1000/frame_ms:12.3f}")


 angle_deg  pulse_us  frame_ms  frequency_hz
         0       500     20.500        48.780
        80      1380     20.380        49.068
       180      2480     20.480        48.828


### 4.3 Interpretation

The equation, the Arduino C++ implementation, and the numerical output agree: the requested pulse widths remain inside a frame of approximately 20.4 ms, corresponding to roughly 49 Hz. The values are analytical checks of the source constants, not oscilloscope measurements.


## 5. Joystick Acquisition and Command Logic

`analogRead()` yields a 10-bit value $u\in[0,1023]$. The source implements a three-region relay rather than proportional control:

$$
c(u)=\begin{cases}-1,&u<50\\0,&50\leq u\leq1000\\+1,&u>1000.\end{cases}
$$

The wide neutral region suppresses drift but requires the stick to approach an endpoint before motion begins. Command speed is set by the selected angle increment and blocking delay, not by joystick displacement.


### 5.1 Arduino C++ implementation

The right joystick X axis controls the base with the threshold function and applies the calibrated limits before commanding the servo:

```cpp
if (x1 < 50)
{
  pos1 = max(pos1 - 1, 1);
  myservo1.write(pos1);
  delay(5);
}

if (x1 > 1000)
{
  pos1 = min(pos1 + 1, 180);
  myservo1.write(pos1);
  delay(5);
}
```

The other three joystick axes use the same relay structure with their own increments, limits, and servo assignments.


### 5.2 Visual result

![Joystick threshold response](../assets/plots/joystick-threshold.svg)

*Figure 7. Source-derived command response across the complete 10-bit ADC range.*


### 5.3 Interpretation

The graph confirms that almost the entire ADC range is neutral: only readings below 50 or above 1000 generate motion. The response is directional rather than proportional, so pushing farther does not increase speed. Applying the angle bound before `Servo.write()` prevents an out-of-range command on the limiting iteration. A one-degree update followed by `delay(5)` gives a nominal software command rate of at most 200 degrees per second; this is not a measured shaft velocity.


## 6. Forward Kinematics and Reachable Workspace

A servo command is not automatically equal to a mechanical joint angle. Horn orientation, linkage geometry, and assembly offsets require a calibrated conversion

$$
\phi_i=s_i(q_i-q_{i,0})+\phi_{i,0}, \qquad s_i\in\{-1,+1\},
$$

where $q_i$ is the commanded servo angle, $q_{i,0}$ is its calibrated command reference, $\phi_{i,0}$ is the corresponding physical joint angle, and $s_i$ captures direction. These parameters must be measured on the assembled arm rather than inferred from the drawing.

For an idealized two-link arm with base yaw $\phi_1$, shoulder angle $\phi_2$, elbow angle $\phi_3$, link lengths $L_1$ and $L_2$, and shoulder height $h$, define the horizontal radial reach

$$
r=L_1\cos\phi_2+L_2\cos(\phi_2+\phi_3).
$$

The approximate gripper position is then

$$
x=r\cos\phi_1, \qquad y=r\sin\phi_1,
$$

$$
z=h+L_1\sin\phi_2+L_2\sin(\phi_2+\phi_3).
$$

Servo 4 controls gripper opening and is therefore excluded from this position model. Before joint limits are applied, the planar distance from the shoulder obeys

$$
|L_1-L_2|\leq\sqrt{r^2+(z-h)^2}\leq L_1+L_2.
$$

The real workspace is a smaller subset because each calibrated joint is constrained by $\phi_i^{\min}\leq\phi_i\leq\phi_i^{\max}$, and because the acrylic linkage is not a perfect unconstrained serial mechanism. Consequently, this model is a geometric approximation until link lengths, zero offsets, and physical joint angles are measured.


### 6.1 Arduino C++ implementation

The following C++ helper implements the idealized equations. It is an analytical extension for the notebook and is not yet called by the control sketches. Meaningful inputs require measured link lengths and calibrated physical joint angles.

```cpp
struct CartesianPoint
{
  float x;
  float y;
  float z;
};

CartesianPoint forwardKinematics(float phi1Deg, float phi2Deg,
                                 float phi3Deg, float L1,
                                 float L2, float h)
{
  float phi1 = radians(phi1Deg);
  float phi2 = radians(phi2Deg);
  float phi3 = radians(phi3Deg);
  float radial = L1 * cos(phi2) + L2 * cos(phi2 + phi3);

  return {radial * cos(phi1),
          radial * sin(phi1),
          h + L1 * sin(phi2) + L2 * sin(phi2 + phi3)};
}
```


### 6.2 Supplemental calculation

The Python helper below mirrors the C++ equations for notebook calculations. It intentionally does not evaluate a physical coordinate until measured calibration values are supplied.


In [ ]:
from math import cos, sin, radians

def servo_to_joint(servo_deg, servo_zero_deg, joint_zero_deg=0.0, direction=1):
    """Convert a servo command to a calibrated joint angle in degrees."""
    return joint_zero_deg + direction * (servo_deg - servo_zero_deg)

def forward_kinematics(phi1_deg, phi2_deg, phi3_deg, L1, L2, h=0.0):
    """Idealized base-yaw plus planar two-link forward kinematics."""
    phi1, phi2, phi3 = map(radians, (phi1_deg, phi2_deg, phi3_deg))
    radial = L1 * cos(phi2) + L2 * cos(phi2 + phi3)
    x = radial * cos(phi1)
    y = radial * sin(phi1)
    z = h + L1 * sin(phi2) + L2 * sin(phi2 + phi3)
    return x, y, z

# Supply measured L1, L2, h and calibrated joint angles before evaluation.


### 6.3 Visual result

![Idealized normalized workspace](../assets/plots/normalized-workspace.svg)

*Figure 8. Normalized geometric workspace implied by the two-link equations. This is an analytical illustration, not a measurement of the physical arm.*


### 6.4 Interpretation

The visual result reflects the theoretical reach bounds: the sum of the link lengths sets the outer boundary and their absolute difference sets the inner boundary. Base rotation sweeps this planar region around the vertical axis. The physical robot will reach only a subset because servo limits, horn offsets, the acrylic linkage, and self-collision are not represented. Therefore the equations provide a calibration framework, not evidence of Cartesian accuracy.


## 7. Pose Representation and Capture

A saved pose is the commanded joint vector

$$
\mathbf q_k=[q_{1,k},q_{2,k},q_{3,k},q_{4,k}]^T.
$$

The source records `Servo.read()` values. In the Arduino Servo library this is the last commanded angle, not encoder feedback. Consequently, memory is open-loop: a stalled or displaced joint is not detected.


### 7.1 Arduino C++ implementation

```cpp
if (nextPose < MAX_POSES)
{
  poses[nextPose][0] = myservo1.read();
  poses[nextPose][1] = myservo2.read();
  poses[nextPose][2] = myservo3.read();
  poses[nextPose][3] = myservo4.read();
  nextPose++;
  poseCount = nextPose;
}
```

The edge-detected right-button event executes this block once per press. `MAX_POSES` bounds the write index before any array access.


### 7.2 State result

The following trace illustrates the bounded state changes; the angle vectors are the same synthetic poses used by the timing example and are not measurements.

| Event | `nextPose` | `poseCount` | Stored or replayed state |
|---|---:|---:|---|
| Boot | 0 | 0 | memory empty |
| Capture $\mathbf q_A=[80,60,130,0]^T$ | 1 | 1 | slot 0 contains $\mathbf q_A$ |
| Capture $\mathbf q_B=[20,100,60,90]^T$ | 2 | 2 | slots 0 and 1 contain the sequence |
| Replay | 0 | 2 | visit slot 0, then slot 1; recording index resets |


### 7.3 Interpretation

The state trace matches the bounded-array model: each accepted button edge adds one four-angle vector, replay visits only the recorded slots, and a full buffer cannot write past its capacity. Because `Servo.read()` returns the last command, the stored pose describes controller state rather than measured mechanical state. Resetting `nextPose` after replay allows a new recording sequence to overwrite old slots intentionally.


## 8. Sequential and Synchronized Pose Replay

For a transition from pose $k-1$ to $k$, the number of one-degree command steps is

$$N_{\mathrm{seq},k}=\sum_{i=1}^{4}|q_{i,k}-q_{i,k-1}|.$$
With a 5 ms delay per step, the source-level estimate is

$$t_{\mathrm{seq},k}\approx 5\,\text{ms}\,N_{\mathrm{seq},k}.$$

This is an $L_1$ path in joint space because the servos move one after another. It is not synchronized interpolation and does not specify a Cartesian end-effector path.

For a synchronized one-degree scheduler, define the commanded displacement of joint $i$ as

$$
\Delta q_i=q_{i,k}-q_{i,k-1}, \qquad D_i=|\Delta q_i|.
$$

All joints can finish together in

$$
N_{\mathrm{sync},k}=\max_i D_i
$$

scheduler ticks. A discrete linear interpolation that reaches every target exactly is

$$
q_i[n]=q_{i,k-1}+\operatorname{round}\!\left(\frac{n}{N_{\mathrm{sync},k}}\Delta q_i\right),
\qquad n=0,1,\ldots,N_{\mathrm{sync},k}.
$$

At a 5 ms scheduler interval, the ideal source-level duration becomes

$$
t_{\mathrm{sync},k}\approx5\,\text{ms}\max_i|\Delta q_i|.
$$

The theoretical command-time improvement is

$$
\rho_k=\frac{t_{\mathrm{seq},k}}{t_{\mathrm{sync},k}}=\frac{\sum_i|\Delta q_i|}{\max_i|\Delta q_i|}, \qquad 1\leq\rho_k\leq4.
$$

These times describe command scheduling only. Actual motion also depends on servo speed, acceleration, load, supply voltage, and mechanical compliance.


### 8.1 Current Arduino C++ implementation

The repository currently moves one servo completely before starting the next servo:

```cpp
void moveServoTo(Servo &servo, int &position, int target)
{
  while (position != target)
  {
    position += target > position ? 1 : -1;
    servo.write(position);
    delay(5);
  }
}

void playPose(int index)
{
  moveServoTo(myservo1, pos1, poses[index][0]);
  moveServoTo(myservo2, pos2, poses[index][1]);
  moveServoTo(myservo3, pos3, poses[index][2]);
  moveServoTo(myservo4, pos4, poses[index][3]);
}
```

A synchronized implementation would evaluate the interpolation equation for all four targets on each scheduler tick. That scheduler is analyzed here but is not presented as current repository behavior.


### 8.2 Numerical and visual result

The Python cell computes the command-step counts directly from the two illustrative pose vectors.


In [2]:
# Illustrative poses, not experimental measurements
pose_a = (80, 60, 130, 0)
pose_b = (20, 100, 60, 90)
deltas = [abs(b - a) for a, b in zip(pose_a, pose_b)]
sequential_steps = sum(deltas)
synchronized_steps = max(deltas)
tick_seconds = 0.005
print('Illustrative transition only')
print('joint deltas:', deltas, 'degrees')
print('sequential scheduler steps:', sequential_steps)
print('synchronized scheduler steps:', synchronized_steps)
print(f'estimated sequential command time: {sequential_steps * tick_seconds:.3f} s')
print(f'ideal synchronized command time: {synchronized_steps * tick_seconds:.3f} s')
print(f'theoretical scheduling improvement: {sequential_steps / synchronized_steps:.3f}x')


Illustrative transition only
joint deltas: [60, 40, 70, 90] degrees
sequential scheduler steps: 260
synchronized scheduler steps: 90
estimated sequential command time: 1.300 s
ideal synchronized command time: 0.450 s
theoretical scheduling improvement: 2.889x


![Sequential and synchronized joint trajectories](../assets/plots/motion-timing-comparison.svg)

*Figure 9. Joint-command trajectories for the illustrative transition. The left panel reproduces current sequential scheduling; the right panel shows the analytical synchronized alternative.*


### 8.3 Interpretation

The numerical output and plot agree with the theory. The current sequence spends 260 ticks, or approximately 1.30 s of programmed delay, because the four absolute angle changes are added. The synchronized alternative finishes after the largest change, 90 ticks or 0.45 s, so all four commands arrive together with 2.889 times fewer scheduler ticks. These are command-scheduling estimates only; actual servo motion can be slower and depends on load, voltage, acceleration, and linkage compliance.

The current C++ implementation remains useful for teaching because its control flow is simple and every target is reached exactly. Its blocking `while` loops, however, prevent responsive joystick input and simultaneous motion during replay.


## 9. Development Progression

| Stage | Sketches | Engineering purpose |
|---|---|---|
| Calibration | `hardware_tests/servo_*` | Establish safe assembly angles and low-level pulse behavior |
| Basic control | `servo_basic_test`, `servo_control` | Validate Servo library output and a fixed sequence |
| Input acquisition | `read_joystick_values` | Observe raw analog and button states |
| Manual control | `dual_joystick_control` | Map joystick regions to bounded joint increments |
| Single memory | `memory_single_pose` | Capture and return to one commanded pose |
| Sequence memory | `memory_multiple_poses` | Replay as many as 10 poses once |
| Loop memory | `memory_with_loop` | Repeat as many as 20 poses until a stop command |


## 10. Results and Media

![20-second physical robot-arm demonstration](../assets/media/robot-arm-demo.gif)

*Figure 10. Physical evidence from the supplied demonstration video.*

[Open the original MP4 demonstration](../assets/media/robot-arm-demo.mp4).

| Claim | Classification | Basis |
|---|---|---|
| Servo and joystick control paths exist | Source-verified | Arduino sketches |
| A physical arm was assembled | Physical evidence | Supplied video |
| Vendor examples produced the described actions | Vendor-reported | Keyestudio test-result sections |
| Pulse frequency is approximately 49 Hz | Calculated | Low-level pulse code |
| Example transition takes 1.30 s sequentially versus 0.45 s with an ideal synchronized scheduler | Illustrative | Synthetic poses above |
| Cartesian position can be estimated after geometric calibration | Analytical model | Symbolic forward-kinematics equations above |
| Accuracy, payload, current, and repeatability | Unknown | No project measurements supplied |

No synthetic calculation in this notebook is presented as an experimental result.


## 11. Known Limitations and Uncertainties

1. **No physical feedback.** Commanded and achieved angles can differ under load or obstruction.
2. **Blocking replay.** Motion functions monopolize the loop until a target is reached.
3. **Sequential joints.** The end-effector path is not explicitly controlled.
4. **Calibration dependence.** Limits must match the actual horn and linkage assembly.
5. **Asymmetric gripper increments.** Open and close command rates differ.
6. **Servo 3 physical-direction ambiguity.** The repository sketches agree on numerical angle direction, but the vendor control table describes the resulting extend/retract motion in the opposite way. Physical horn and linkage orientation must determine the final labels.
7. **No quantitative experiment dataset.** Accuracy, repeatability, current, payload, and speed remain unmeasured.
8. **Board-specific build verification pending.** All 10 sketches pass host-side C++ syntax validation with a minimal Arduino API stub, but `arduino-cli` was unavailable for an Arduino Uno toolchain compile.


## 12. Engineering Conclusions

The project demonstrates a coherent learning progression from actuator calibration to an open-loop teach-and-repeat controller. The pose-vector model is appropriate for a low-cost arm without sensors, and the revised bounds and button-event logic remove the most important memory-safety and replay defects.

The symbolic forward-kinematics model now provides a framework for converting calibrated joint angles into an approximate Cartesian position. Its numerical use requires measured link lengths, servo zero offsets, joint directions, and safe angle ranges. Those measurements should precede inverse kinematics or claims about Cartesian accuracy.

The timing comparison also shows why synchronized interpolation is the next logical control improvement: it can reduce command scheduling time while moving all joints together. Physical timing and trajectory quality must still be measured because the analytical estimate does not model servo acceleration, load, supply voltage, or linkage compliance.


## 13. Reproduction Checklist

1. Install Arduino IDE and the CP2102 driver if the board is not detected.
2. Select an Arduino Uno-compatible target and the correct serial port.
3. Verify one calibration sketch at a time before attaching the complete linkage.
4. Connect external servo power through the driver shield and use a common ground.
5. Upload one independent sketch directory.
6. Use the Serial Monitor at 9600 baud where status output is implemented.
7. Record any hardware-specific limit or direction change before editing the mapping.

Notebook calculations require only a standard Jupyter environment and Python's standard library.
